<a href="https://colab.research.google.com/github/AhmadFebriyan5/Studi-Kasus_Machine-Learning/blob/main/6_Attributing_Direct_Drivers_to_Deforestation_Alerts_with_Satellite_Imagery_and_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klasifikasi Pemicu Deforestasi dengan CNN

**Studi kasus:** mengklasifikasikan gangguan hutan ke dalam enam kategori menggunakan data multi-band bergaya Sentinel-1 dan Sentinel-2.



## 1. Persiapan

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

LABELS = [
    "Pertanian skala kecil",
    "Pertanian skala besar",
    "Pembangunan jalan",
    "Pembalakan selektif",
    "Penambangan",
    "Kebakaran/gangguan alami"
]

print("TensorFlow:", tf.__version__)

## 2. Dataset demo multi-band

In [ ]:
# Patch sintetis: 16x16 piksel, 6 kanal contoh (radar + optik)
# Ganti bagian ini dengan data citra satelit berlabel untuk eksperimen nyata.
N, H, W, C = 1200, 16, 16, 6
X = np.random.normal(0, 0.25, (N, H, W, C)).astype("float32")
y = np.repeat(np.arange(len(LABELS)), N // len(LABELS))
y = y[:N]
X = X[:len(y)]

# Pola buatan berbeda per kelas agar model bisa belajar pada data demo
for i, label in enumerate(y):
    X[i, :, :, i % C] += 0.8
    if label == 2:  # pola garis sederhana untuk demo jalan
        X[i, H // 2 - 1:H // 2 + 1, :, 2] += 1.0
    elif label == 3:  # beberapa area kecil
        X[i, 3:7, 3:7, 3] += 0.9
        X[i, 10:13, 10:13, 3] += 0.9
    elif label == 4:  # blok area
        X[i, 4:12, 4:12, 4] += 0.8
    elif label == 5:  # gangguan tersebar
        X[i, ::3, ::3, 5] += 0.9

X = np.clip(X, -1, 2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

## 3. Bangun dan latih CNN

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(H, W, C)),
    tf.keras.layers.Conv2D(16, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(len(LABELS), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=8,
    batch_size=32,
    verbose=1
)

## 4. Evaluasi model

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy (data sintetis): {test_acc:.3f}")

pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test, pred, target_names=LABELS, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred, display_labels=LABELS, xticks_rotation=45
)
plt.tight_layout()
plt.show()

## 5. Prediksi satu patch

In [ ]:
idx = 0
probs = model.predict(X_test[idx:idx+1], verbose=0)[0]
predicted_class = int(np.argmax(probs))

plt.imshow(X_test[idx, :, :, :3])
plt.title(f"Prediksi: {LABELS[predicted_class]} | Aktual: {LABELS[y_test[idx]]}")
plt.axis("off")
plt.show()

print("Probabilitas per kelas:")
for name, prob in zip(LABELS, probs):
    print(f"{name}: {prob:.3f}")